In [1]:
pip install imutils

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pygame8888888888888888

ERROR: Could not find a version that satisfies the requirement pygame8888888888888888 (from versions: none)
ERROR: No matching distribution found for pygame8888888888888888
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import dlib
import cv2
import threading
from threading import Thread
import imutils
from imutils import face_utils
from scipy.spatial import distance as dist
import pygame
import serial
import time

# === Serial Setup ===
try:
    ser = serial.Serial('/dev/ttyUSB0', 115200)  # Replace with your port
    print("[INFO] Serial connected.")
except Exception as e:
    print(f"[WARNING] Serial port not connected: {e}")
    ser = None

def send_serial_command(command_hex):
    if ser and ser.is_open:
        ser.write(bytes.fromhex(command_hex))
        time.sleep(0.05)

# === Alarm Sound ===
def sound_alarm():
    pygame.mixer.init()
    pygame.mixer.music.load("D:/NewDesk/A.wav")
    pygame.mixer.music.play()

# === Eye Aspect Ratio (EAR) ===
def eye_aspect_ratio(eye):
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[3])
    C = dist.euclidean(eye[0], eye[3])
    return (A + B) / (2.0 * C)

# === Thresholds ===
EYE_AR_THRESH = 0.3
EYE_AR_CONSEC_FRAMES = 30

COUNTER = 0
ALARM_ON = False
SERIAL_STATE = False  # Keep track if serial is ON or OFF to avoid repeats

# === Dlib Model ===
predictor_path = '/home/gott/Downloads/shape_predictor_68_face_landmarks.dat'
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(predictor_path)

(lStart, lEnd) = face_utils.FACIAL_LANDMARKS_IDXS["left_eye"]
(rStart, rEnd) = face_utils.FACIAL_LANDMARKS_IDXS["right_eye"]

# === Camera Setup ===
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to capture frame. Check your camera.")
        break

    frame = imutils.resize(frame, width=450)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    rects = detector(gray, 0)

    for rect in rects:
        shape = predictor(gray, rect)
        shape = face_utils.shape_to_np(shape)

        leftEye = shape[lStart:lEnd]
        rightEye = shape[rStart:rEnd]
        leftEAR = eye_aspect_ratio(leftEye)
        rightEAR = eye_aspect_ratio(rightEye)
        ear = (leftEAR + rightEAR) / 2.0

        leftEyeHull = cv2.convexHull(leftEye)
        rightEyeHull = cv2.convexHull(rightEye)
        cv2.drawContours(frame, [leftEyeHull], -1, (0, 255, 0), 1)
        cv2.drawContours(frame, [rightEyeHull], -1, (0, 255, 0), 1)

        if ear < EYE_AR_THRESH:
            COUNTER += 1

            if COUNTER >= EYE_AR_CONSEC_FRAMES:
                if not ALARM_ON:
                    ALARM_ON = True
                    d = Thread(target=sound_alarm)
                    d.daemon = True
                    d.start()

                cv2.putText(frame, "DROWSINESS ALERT!", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

                # Send serial ON command once
                if not SERIAL_STATE:
                    send_serial_command("3A0101020003000400")  # <-- your ON command
                    SERIAL_STATE = True

        else:
            COUNTER = 0
            ALARM_ON = False
            if SERIAL_STATE:
                send_serial_command("3A0100020003000400")  # <-- your OFF command
                SERIAL_STATE = False

        cv2.putText(frame, f"EAR: {ear:.2f}", (300, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        send_serial_command("3A0100020003000400")  # OFF on exit
        break

# === Cleanup ===
cap.release()
cv2.destroyAllWindows()
if ser and ser.is_open:
    ser.close()


pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
[INFO] Serial connected.


Exception in thread Thread-5 (sound_alarm):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/home/gott/openvino_env/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_4591/1551097210.py", line 29, in sound_alarm
pygame.error: No file 'D:/NewDesk/A.wav' found in working directory '/home/gott/openvino_notebooks/notebooks'.
Exception in thread Thread-6 (sound_alarm):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/home/gott/openvino_env/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **s